In [1]:
from sentence_transformers import SentenceTransformer
import chromadb
import ollama

# ==========================================
# CONFIG
# ==========================================
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"
DB_PATH = "./db_khoa_cntt"
COLLECTION_NAME = "tai_lieu_khoa_cntt"
OLLAMA_MODEL = "gemma3:4b"
TOP_K = 5

# ==========================================
# LOAD EMBEDDING MODEL
# ==========================================
print("Đang tải embedding model...")
model = SentenceTransformer(EMBEDDING_MODEL)

# ==========================================
# LOAD CHROMADB
# ==========================================
print("Đang mở Vector Database...")
chroma_client = chromadb.PersistentClient(path=DB_PATH)
collection = chroma_client.get_collection(COLLECTION_NAME)

# ==========================================
# QUERY
# ==========================================
query = input("\nNhập câu hỏi của bạn: ")

# E5 cần prefix query:
formatted_query = "query: " + query

# ==========================================
# EMBED QUERY
# ==========================================
print("\nĐang embedding câu hỏi...")
query_embedding = model.encode(
    formatted_query,
    normalize_embeddings=True
).tolist()

# ==========================================
# RETRIEVE
# ==========================================
print("\nĐang tìm kiếm thông tin liên quan...")

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=TOP_K,
    include=["documents", "distances", "metadatas"]
)

documents = results["documents"][0]
distances = results["distances"][0]
metadatas = results["metadatas"][0]

# ==========================================
# SHOW RETRIEVED CHUNKS
# ==========================================
print("\n====================")
print(f"TOP {TOP_K} CHUNKS")
print("====================\n")

context_blocks = []

for i, (doc, distance, meta) in enumerate(zip(documents, distances, metadatas), 1):
    similarity = 1 - distance
    source = meta.get("source_url", "Không rõ nguồn")

    print(f"--- Rank {i} ---")
    print(f"Similarity: {similarity:.4f}")
    print(f"Source: {source}")
    print(doc[:1000])
    print("\n")

    context_blocks.append(
        f"[Tài liệu {i}]\nNguồn: {source}\nNội dung:\n{doc}"
    )

context = "\n\n".join(context_blocks)

# ==========================================
# PROMPT FOR GEMMA
# ==========================================
prompt = f"""
Bạn là chatbot tư vấn của Khoa Công nghệ Thông tin HCMUTE.

Quy tắc:
- Chỉ trả lời dựa trên Context.
- Trả lời đúng trọng tâm câu hỏi.
- Không lan man.
- Nếu Context không có thông tin, trả lời: "Dữ liệu hiện tại chưa có thông tin này."
- Không bịa thông tin ngoài Context.

Context:
{context}

Câu hỏi:
{query}

Câu trả lời:
"""

# ==========================================
# GENERATE ANSWER WITH OLLAMA GEMMA
# ==========================================
print("\nĐang sinh câu trả lời bằng Gemma3:4b...\n")

response = ollama.chat(
    model=OLLAMA_MODEL,
    messages=[
        {"role": "user", "content": prompt}
    ]
)

answer = response["message"]["content"]

print("====================")
print("CÂU TRẢ LỜI")
print("====================")
print(answer)

Đang tải embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Đang mở Vector Database...

Đang embedding câu hỏi...

Đang tìm kiếm thông tin liên quan...

TOP 5 CHUNKS

--- Rank 1 ---
Similarity: 0.8572
Source: http://fit.hcmute.edu.vn/?ArticleId=01a97601-15dc-4f8b-8bd4-8a062e7c3f73
. Sau hơn 20 năm kể từ khóa đầu tiên, Khoa CNTT đã và đang tiếp tục chứng tỏ là một trong những khoa đi đầu trong đổi mới và nâng cao chất lượng dạy học, đào tạo ra những kĩ sư CNTT chất lượng.


--- Rank 2 ---
Similarity: 0.8509
Source: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=df646c13-b76f-444d-a1d1-95ccdfd50b47
| **2** | **2** | Khám phá tri thức, giải quyết vấn đề, tư duy hệ thống, tư duy sáng tạo và kỹ năng chuyên môn trong lĩnh vực CNTT |
| **3** | **3** | Làm việc nhóm và giao tiếp hiệu quả |
| **4** | **4** | Hình thành ý tưởng, thiết kế, triển khai, vận hành hệ thống CNTT và có kiến thức về lãnh đạo và kinh doanh trong kỹ thuật. |  
**Link:**


--- Rank 3 ---
Similarity: 0.8422
Source: http://fit.hcmute.edu.vn/Default.aspx?ArticleId=bafd29bd-264e-4a5d-